# 09 · Gold — metrics, metric view, Genie agent

Build the numbers Genie should answer from **PDF-extracted** fields (not the synthetic tables in notebook 01).

1. `gold_clinic_invoices` — one row per invoice PDF
2. `gold_suspected_pdf_duplicates` — same pet + clinic + visit_date, more than one invoice
3. Metric view `gold_pdf_invoice_metrics`
4. Comments + paste-ready Genie instructions


In [ ]:
%run ./00_setup


In [ ]:
spark.sql("""
CREATE OR REPLACE TABLE gold_clinic_invoices
COMMENT 'Invoice facts extracted from pet-claim PDFs'
AS
SELECT
  claim_number,
  policy_number,
  pet_name,
  species,
  breed,
  member_name,
  clinic_name,
  visit_date,
  invoice_number,
  billed_amount,
  diagnosis,
  recommended_decision,
  file_name
FROM silver_claim_documents_typed
WHERE doc_type = 'clinic_invoice'
""")

spark.sql("""
CREATE OR REPLACE TABLE gold_claim_documents
COMMENT 'All typed PDF documents for Genie (FNOL, invoice, adjuster, policy)'
AS
SELECT * FROM silver_claim_documents_typed
""")

spark.sql("""
CREATE OR REPLACE TABLE gold_suspected_pdf_duplicates
COMMENT 'Invoices that share pet + clinic + visit_date (workshop duplicate definition)'
AS
WITH d AS (
  SELECT pet_name, clinic_name, visit_date, count(*) AS n_invoices
  FROM gold_clinic_invoices
  WHERE pet_name IS NOT NULL AND clinic_name IS NOT NULL AND visit_date IS NOT NULL
  GROUP BY pet_name, clinic_name, visit_date
  HAVING count(*) > 1
)
SELECT i.*, d.n_invoices
FROM gold_clinic_invoices i
JOIN d
  ON i.pet_name = d.pet_name
 AND i.clinic_name = d.clinic_name
 AND i.visit_date = d.visit_date
""")

display(spark.sql("SELECT * FROM gold_clinic_invoices ORDER BY visit_date, claim_number"))
display(spark.sql("SELECT * FROM gold_suspected_pdf_duplicates"))


In [ ]:
spark.sql(f"""
CREATE OR REPLACE VIEW gold_pdf_invoice_metrics
WITH METRICS LANGUAGE YAML
COMMENT 'Metric view over PDF-extracted clinic invoices'
AS $$
version: 0.1
source: {CATALOG}.{SCHEMA}.gold_clinic_invoices
filter: billed_amount IS NOT NULL
dimensions:
  - name: Clinic
    expr: clinic_name
  - name: Diagnosis
    expr: diagnosis
  - name: Pet
    expr: pet_name
  - name: Visit date
    expr: visit_date
  - name: Claim
    expr: claim_number
measures:
  - name: Invoice count
    expr: COUNT(1)
  - name: Total billed
    expr: SUM(billed_amount)
  - name: Average billed
    expr: AVG(billed_amount)
$$
""")

display(spark.sql("""
SELECT
  `Clinic` AS clinic,
  MEASURE(`Invoice count`) AS invoice_count,
  MEASURE(`Total billed`) AS total_billed,
  MEASURE(`Average billed`) AS avg_billed
FROM gold_pdf_invoice_metrics
GROUP BY `Clinic`
ORDER BY total_billed DESC
"""))


In [ ]:
spark.sql("""
ALTER TABLE gold_clinic_invoices ALTER COLUMN pet_name COMMENT 'Pet name extracted from the invoice PDF'
""")
spark.sql("""
ALTER TABLE gold_clinic_invoices ALTER COLUMN clinic_name COMMENT 'Veterinary clinic extracted from the invoice PDF'
""")
spark.sql("""
ALTER TABLE gold_clinic_invoices ALTER COLUMN visit_date COMMENT 'Visit date. Duplicate key is pet_name + clinic_name + visit_date'
""")
spark.sql("""
ALTER TABLE gold_clinic_invoices ALTER COLUMN billed_amount COMMENT 'Total billed amount from the invoice, USD'
""")
spark.sql("""
ALTER TABLE gold_suspected_pdf_duplicates SET TBLPROPERTIES (
  'comment' = 'Rows that share pet + clinic + visit_date. Max at Pine Vet Clinic on 2026-08-03 is the planted pair.'
)
""")
print("comments applied")


### Enable a Genie agent

In the workspace: **New → Genie space** (or Genie Agent). Add:

- `{catalog}.{schema}.gold_clinic_invoices`
- `{catalog}.{schema}.gold_claim_documents`
- `{catalog}.{schema}.gold_suspected_pdf_duplicates`
- `{catalog}.{schema}.gold_pdf_invoice_metrics`

Paste the instructions from the next cell. Then run the benchmark questions.


In [ ]:
instructions = f"""
You are a pet-insurance claims analyst for a workshop. Answer only from the attached Unity Catalog objects in {CATALOG}.{SCHEMA}.

Tables
- gold_clinic_invoices: one row per clinic invoice PDF. Grain is invoice_number.
- gold_claim_documents: every parsed PDF (FNOL, invoice, adjuster, policy) with doc_type.
- gold_suspected_pdf_duplicates: invoices that share pet_name + clinic_name + visit_date.
- gold_pdf_invoice_metrics: metric view. Use MEASURE(`Invoice count`), MEASURE(`Total billed`), MEASURE(`Average billed`).

Duplicate definition (do not invent another)
A duplicate is the same pet_name, clinic_name, and visit_date on more than one invoice.
Invoice number is not the key. CLM-PET-00044 and CLM-PET-00045 are the planted pair (Max, Pine Vet Clinic, 2026-08-03).
CLM-PET-00088 is a waiting-period denial, not a duplicate.

Rules
- If a question cannot be answered from these objects, say so.
- Prefer gold_suspected_pdf_duplicates when asked about duplicates.
- Money is USD. Do not mix in the synthetic tables from the Genie Code lab unless the user asks.
""".strip()

print(instructions)


### Benchmark questions


In [ ]:
qs = [
    "How many clinic invoices did we extract from PDFs, and what is total billed?",
    "Which invoices share the same pet, clinic, and visit date?",
    "What did Pine Vet Clinic bill for Max, and how many claim numbers are involved?",
    "Is Bella's Maple Emergency Vets invoice a duplicate?",
    "Break down total billed by clinic using the metric view.",
]
for i, q in enumerate(qs, 1):
    print(f"{i}. {q}")
